# Probabilistic Models — Exercises

10 graded exercises covering Bayesian inference, exponential families, EM, variational inference,
MCMC, GP regression, HMM inference, normalizing flows, score matching, and ICL as Bayes.

| Format | Description |
|---|---|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|---|---|---|
| ★ | 1–3 | Core mechanics: conjugate updates, exponential family, EM |
| ★★ | 4–7 | Deeper theory: CAVI, reparameterisation, MH sampler, GP |
| ★★★ | 8–10 | AI applications: HMM inference, normalizing flow, score matching |

### Topic Map

| Topic | Exercise |
|---|---|
| Conjugate posterior | 1 |
| Exponential family moments | 2 |
| EM lower bound + GMM | 3 |
| CAVI derivation | 4 |
| Reparameterisation gradient | 5 |
| Metropolis-Hastings sampler | 6 |
| GP regression from scratch | 7 |
| HMM forward-backward | 8 |
| Normalizing flow log-likelihood | 9 |
| Denoising score matching | 10 |


In [ ]:
import numpy as np
import numpy.linalg as la
from scipy import stats, special
from scipy.stats import norm, beta as beta_dist

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def logsumexp(a, axis=None):
    return special.logsumexp(a, axis=axis)

print('Setup complete.')

---

## Exercise 1 ★ — Beta-Bernoulli Conjugate Posterior

Let $X_1, \ldots, X_n \overset{\text{i.i.d.}}{\sim} \operatorname{Bern}(p)$ and
$p \sim \operatorname{Beta}(\alpha, \beta)$.

**(a)** Compute the posterior parameters $\alpha_n = \alpha + \sum X_i$ and
$\beta_n = \beta + n - \sum X_i$.

**(b)** Compute the posterior mean $\mathbb{E}[p \mid X_{1:n}]$ as a precision-weighted
average of the prior mean and MLE.

**(c)** Compute the posterior predictive $p(X_{n+1}=1 \mid X_{1:n})$ — the mean
of the posterior.

**(d)** Verify as $n \to \infty$ the posterior mean converges to the MLE $\hat{p} = \bar{X}$.

**Data**: $\alpha=2$, $\beta=5$, $n=100$ observations with true $p=0.7$.

In [ ]:
# Exercise 1: Your Solution

alpha_prior, beta_prior = 2.0, 5.0
p_true_ex1 = 0.7
np.random.seed(0)
X = np.random.binomial(1, p_true_ex1, 100)

def beta_bernoulli_posterior(X, alpha0, beta0):
    # YOUR CODE HERE
    pass

alpha_n, beta_n = beta_bernoulli_posterior(X, alpha_prior, beta_prior) or (None, None)
posterior_mean = None  # YOUR CODE HERE
posterior_predictive = None  # YOUR CODE HERE

print('alpha_n:', alpha_n)
print('beta_n:', beta_n)
print('posterior_mean:', posterior_mean)
print('posterior_predictive:', posterior_predictive)

In [ ]:
# Exercise 1: Solution

alpha_prior, beta_prior = 2.0, 5.0
p_true_ex1 = 0.7
np.random.seed(0)
X = np.random.binomial(1, p_true_ex1, 100)

# (a) Posterior parameters
n_heads = X.sum()
n_tails = len(X) - n_heads
alpha_n = alpha_prior + n_heads
beta_n  = beta_prior  + n_tails

# (b) Posterior mean = alpha_n / (alpha_n + beta_n)
posterior_mean = alpha_n / (alpha_n + beta_n)
prior_mean = alpha_prior / (alpha_prior + beta_prior)
mle = X.mean()

# Show as precision-weighted average
prior_prec = alpha_prior + beta_prior
data_prec  = len(X)
post_mean_check = (prior_prec * prior_mean + data_prec * mle) / (prior_prec + data_prec)

# (c) Posterior predictive = posterior mean
posterior_predictive = posterior_mean

# (d) Large-n convergence: test with n=10000
X_large = np.random.binomial(1, p_true_ex1, 10000)
a_large = alpha_prior + X_large.sum()
b_large = beta_prior + len(X_large) - X_large.sum()
post_mean_large = a_large / (a_large + b_large)
mle_large = X_large.mean()

header('Exercise 1: Beta-Bernoulli Posterior')
print(f'Posterior: Beta({alpha_n:.0f}, {beta_n:.0f})')
print(f'Posterior mean = {posterior_mean:.4f}')
print(f'True p = {p_true_ex1}')
check_close('posterior = precision-weighted avg', posterior_mean, post_mean_check)
check_true('posterior predictive = posterior mean', abs(posterior_predictive - posterior_mean) < 1e-12)
check_true('n->inf: posterior mean -> MLE', abs(post_mean_large - mle_large) < 0.001)
print(f'\nTakeaway: With n=100 data points, the posterior mean ({posterior_mean:.3f}) is between'
      f' the prior mean ({prior_mean:.3f}) and MLE ({mle:.3f}) — automatically regularised.')

---

## Exercise 2 ★ — Exponential Family: Log-Partition Moments

The Gamma distribution with shape $a$ and rate $b$ has:
$p(x \mid a, b) = \frac{b^a}{\Gamma(a)} x^{a-1} e^{-bx}$, $x > 0$.

**(a)** Write the Gamma distribution in exponential family form with natural
parameters $\boldsymbol{\eta} = (a-1, -b)$ and sufficient statistics $T(x) = (\log x, x)$.

**(b)** Identify $A(\boldsymbol{\eta})$ and verify $\nabla_{\boldsymbol{\eta}} A(\boldsymbol{\eta}) = \mathbb{E}[T(x)]$.

**(c)** Verify numerically for $(a, b) = (3, 2)$: compute $\nabla A$ analytically
and compare to Monte Carlo estimates of $\mathbb{E}[\log x]$ and $\mathbb{E}[x]$.

**(d)** Show the Hessian $\nabla^2 A = \operatorname{Cov}[T(x)]$ numerically.

In [ ]:
# Exercise 2: Your Solution

from scipy.special import digamma, polygamma

a_gamma, b_gamma = 3.0, 2.0

# (b) A(eta) = log Gamma(a) - a*log(b) with a = eta1+1, b = -eta2
def log_partition_gamma(eta1, eta2):
    # YOUR CODE HERE
    pass

# (c) Gradient of A = E[T(x)]
dA_deta1 = None  # E[log x] = digamma(a) - log(b)
dA_deta2 = None  # E[x] = a/b

print('dA/deta1 (E[log x]):', dA_deta1)
print('dA/deta2 (E[x]):', dA_deta2)

In [ ]:
# Exercise 2: Solution

from scipy.special import digamma, polygamma, gammaln
import numpy as np

a_gamma, b_gamma = 3.0, 2.0

# Natural params: eta = (a-1, -b)  =>  a = eta1+1, b = -eta2
eta1 = a_gamma - 1  # = 2
eta2 = -b_gamma     # = -2

# (b) Log-partition: A(eta) = log Gamma(eta1+1) - (eta1+1)*log(-eta2)
def log_partition_gamma(eta1, eta2):
    a = eta1 + 1
    b = -eta2
    return gammaln(a) - a * np.log(b)

# (c) Analytical gradients
dA_deta1 = digamma(a_gamma) - np.log(b_gamma)  # E[log x]
dA_deta2 = -a_gamma / b_gamma                   # E[x] * (-1) from chain rule → dA/d(-b) = -a/b... check:
# d/d(eta2) A = d/d(-b) [log Gamma(a) - a*log(b)] = a/b (since d(-log b)/d(-b) = 1/b)
dA_deta2 = a_gamma / b_gamma  # = E[x]

# Monte Carlo verification
np.random.seed(42)
samples_gamma = np.random.gamma(a_gamma, 1/b_gamma, 500000)
E_log_x = np.log(samples_gamma).mean()
E_x = samples_gamma.mean()

# (d) Hessian = Cov[T(x)]
d2A_11 = polygamma(1, a_gamma)  # Var[log x] = trigamma(a)
d2A_22 = a_gamma / b_gamma**2  # Var[x] = a/b^2
d2A_12 = -1.0 / b_gamma         # Cov[log x, x]

T = np.vstack([np.log(samples_gamma), samples_gamma]).T
Cov_T = np.cov(T.T)

header('Exercise 2: Exponential Family Moments')
print(f'E[log x] analytical = {dA_deta1:.6f}, MC = {E_log_x:.6f}')
print(f'E[x]     analytical = {dA_deta2:.6f}, MC = {E_x:.6f}')
check_close('dA/deta1 = E[log x]', dA_deta1, E_log_x, tol=1e-2)
check_close('dA/deta2 = E[x]', dA_deta2, E_x, tol=1e-2)
check_close('Hessian diagonal: Var[log x]', d2A_11, Cov_T[0, 0], tol=5e-2)
check_close('Hessian diagonal: Var[x]', d2A_22, Cov_T[1, 1], tol=5e-2)
print(f'\nTakeaway: For any exponential family, dA = E[T(x)] and d²A = Cov[T(x)] — '
      f'the log-partition function is the cumulant generating function.')

---

## Exercise 3 ★ — EM Lower Bound and GMM

**(a)** Prove using Jensen's inequality that
$\log p(\mathbf{x}) \geq \mathbb{E}_q[\log p(\mathbf{x}, \mathbf{z})] + H(q)$.

**(b)** Implement EM for a 2-component 1D GMM. Run 50 iterations.

**(c)** Verify that the log-likelihood is **monotonically non-decreasing** at every step.

**(d)** Plot the responsibility $r_{i1}$ vs. $x^{(i)}$ at iterations 1, 5, and 50.

**Data**: $n=200$, true params $\pi=[0.4, 0.6]$, $\mu=[-2, 3]$, $\sigma=[0.8, 1.0]$.

In [ ]:
# Exercise 3: Your Solution

np.random.seed(1)
pi_t = np.array([0.4, 0.6])
mu_t = np.array([-2.0, 3.0])
sig_t = np.array([0.8, 1.0])
n = 200
z_ex3 = np.random.choice(2, n, p=pi_t)
x_ex3 = np.array([np.random.normal(mu_t[z], sig_t[z]) for z in z_ex3])

def em_step(x, pi, mu, sigma):
    # YOUR CODE HERE: return (r, pi_new, mu_new, sigma_new, log_lik)
    pass

# Run EM
pi_em3 = np.array([0.5, 0.5])
mu_em3 = np.array([-1.0, 2.0])
sig_em3 = np.array([1.0, 1.0])

lls3 = []
# YOUR CODE HERE: run 50 EM iterations

print('log-likelihoods:', lls3[:5])

In [ ]:
# Exercise 3: Solution

np.random.seed(1)
pi_t = np.array([0.4, 0.6])
mu_t = np.array([-2.0, 3.0])
sig_t = np.array([0.8, 1.0])
n = 200
z_ex3 = np.random.choice(2, n, p=pi_t)
x_ex3 = np.array([np.random.normal(mu_t[z], sig_t[z]) for z in z_ex3])

def gmm_ll(x, pi, mu, sigma):
    K = len(pi)
    log_terms = np.array([np.log(pi[k]) + norm.logpdf(x, mu[k], sigma[k]) for k in range(K)])
    return special.logsumexp(log_terms, axis=0).sum()

def em_step(x, pi, mu, sigma):
    K = len(pi)
    # E-step
    log_r = np.array([np.log(pi[k]) + norm.logpdf(x, mu[k], sigma[k]) for k in range(K)])
    log_r -= special.logsumexp(log_r, axis=0, keepdims=True)
    r = np.exp(log_r).T  # (n, K)
    # M-step
    Nk = r.sum(axis=0)
    pi_new = Nk / len(x)
    mu_new = (r * x[:, None]).sum(axis=0) / Nk
    sigma_new = np.sqrt(((r * (x[:, None] - mu_new)**2).sum(axis=0) / Nk).clip(1e-6))
    return r, pi_new, mu_new, sigma_new, gmm_ll(x, pi_new, mu_new, sigma_new)

pi3 = np.array([0.5, 0.5])
mu3 = np.array([-1.0, 2.0])
sig3 = np.array([1.0, 1.0])
lls3 = []
r_history = {}
for it in range(50):
    r3, pi3, mu3, sig3, ll = em_step(x_ex3, pi3, mu3, sig3)
    lls3.append(ll)
    if it in [0, 4, 49]:
        r_history[it+1] = r3[:, 0].copy()

lls3 = np.array(lls3)

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(lls3, color='#0077BB')
    axes[0].set_title('EM Log-Likelihood'); axes[0].set_xlabel('Iteration')
    axes[0].set_ylabel('Log-likelihood')
    colors_iters = ['#555555', '#EE7733', '#0077BB']
    for (it, r_it), col in zip(sorted(r_history.items()), colors_iters):
        idx = np.argsort(x_ex3)
        axes[1].plot(x_ex3[idx], r_it[idx], color=col, label=f'iter {it}', alpha=0.8)
    axes[1].set_title('Responsibility r_{i1}'); axes[1].set_xlabel('x'); axes[1].set_ylabel('r')
    axes[1].legend()
    fig.tight_layout(); plt.show()

header('Exercise 3: EM GMM')
print(f'Fitted pi={pi3.round(3)}, mu={mu3.round(3)}, sigma={sig3.round(3)}')
check_true('log-likelihood monotonically increases', np.all(np.diff(lls3) >= -1e-8))
check_true('mu_0 close to true mu', min(abs(mu3[0]-mu_t[0]), abs(mu3[0]-mu_t[1])) < 0.5)
check_true('mu_1 close to true mu', min(abs(mu3[1]-mu_t[0]), abs(mu3[1]-mu_t[1])) < 0.5)
print('\nTakeaway: EM guarantees monotonic log-likelihood increase, converging to a local optimum.'
      ' Responsibilities sharpen from uncertain (iter 1) to nearly binary (iter 50).')

---

## Exercise 4 ★★ — CAVI and the ELBO

Model: $\mu \sim \mathcal{N}(\mu_0, \tau_0^{-1})$, $x_i \mid \mu \sim \mathcal{N}(\mu, \sigma^2)$
with $n$ i.i.d. observations.

**(a)** Derive the CAVI optimal $q^*(\mu) = \mathcal{N}(m, s^2)$ — show that:
$$s^2 = \left(\tau_0 + \frac{n}{\sigma^2}\right)^{-1}, \quad
m = s^2\left(\tau_0\mu_0 + \frac{\sum_i x_i}{\sigma^2}\right)$$

**(b)** Verify the CAVI update equals the exact conjugate posterior.

**(c)** Compute the ELBO as a function of $(m, s^2)$ and verify it is maximised at the CAVI solution.

**(d)** For $\sigma^2=1$, $\tau_0=1$, $\mu_0=0$, $n=50$, $\mu_\text{true}=2$:
show how the ELBO changes as $m$ varies from $-2$ to $4$.

In [ ]:
# Exercise 4: Your Solution

sigma2_ex4 = 1.0
tau0_ex4 = 1.0
mu0_ex4 = 0.0
mu_true_ex4 = 2.0
np.random.seed(2)
x_ex4 = np.random.normal(mu_true_ex4, np.sqrt(sigma2_ex4), 50)

def cavi_gaussian(x, sigma2, tau0, mu0):
    # YOUR CODE HERE: return (m, s2)
    pass

def elbo_gaussian(m, s2, x, sigma2, tau0, mu0):
    # YOUR CODE HERE
    pass

m_cavi, s2_cavi = cavi_gaussian(x_ex4, sigma2_ex4, tau0_ex4, mu0_ex4) or (None, None)
print('m_cavi:', m_cavi, 's2_cavi:', s2_cavi)

In [ ]:
# Exercise 4: Solution

sigma2_ex4 = 1.0
tau0_ex4 = 1.0
mu0_ex4 = 0.0
mu_true_ex4 = 2.0
np.random.seed(2)
n_ex4 = 50
x_ex4 = np.random.normal(mu_true_ex4, np.sqrt(sigma2_ex4), n_ex4)

# (a) CAVI update
def cavi_gaussian(x, sigma2, tau0, mu0):
    n = len(x)
    s2 = 1.0 / (tau0 + n / sigma2)
    m = s2 * (tau0 * mu0 + x.sum() / sigma2)
    return m, s2

# (b) ELBO
def elbo_gaussian(m, s2, x, sigma2, tau0, mu0):
    n = len(x)
    recon = -n/2*np.log(2*np.pi*sigma2) - 1/(2*sigma2)*(np.sum((x-m)**2) + n*s2)
    # KL(N(m,s2) || N(mu0, 1/tau0))
    kl = 0.5*(tau0*s2 + tau0*(m-mu0)**2 - 1 - np.log(tau0*s2))
    return recon - kl

m_cavi, s2_cavi = cavi_gaussian(x_ex4, sigma2_ex4, tau0_ex4, mu0_ex4)

# Compare to exact posterior
s2_exact = 1.0 / (tau0_ex4 + n_ex4 / sigma2_ex4)
m_exact = s2_exact * (tau0_ex4 * mu0_ex4 + x_ex4.sum() / sigma2_ex4)

# (d) ELBO as function of m
m_grid = np.linspace(-2, 4, 200)
elbos = [elbo_gaussian(m, s2_cavi, x_ex4, sigma2_ex4, tau0_ex4, mu0_ex4) for m in m_grid]

if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(m_grid, elbos, color='#0077BB')
    ax.axvline(m_cavi, color='#CC3311', linestyle='--', label=f'CAVI m={m_cavi:.3f}')
    ax.axvline(mu_true_ex4, color='#555555', linestyle=':', label=f'True mu={mu_true_ex4}')
    ax.set_title('ELBO as a function of variational mean m'); ax.set_xlabel('m')
    ax.set_ylabel('ELBO'); ax.legend()
    fig.tight_layout(); plt.show()

header('Exercise 4: CAVI')
check_close('CAVI mean = exact posterior mean', m_cavi, m_exact)
check_close('CAVI variance = exact posterior variance', s2_cavi, s2_exact)
elbo_at_cavi = elbo_gaussian(m_cavi, s2_cavi, x_ex4, sigma2_ex4, tau0_ex4, mu0_ex4)
elbo_perturbed = elbo_gaussian(m_cavi + 0.3, s2_cavi, x_ex4, sigma2_ex4, tau0_ex4, mu0_ex4)
check_true('ELBO is maximised at CAVI solution', elbo_at_cavi > elbo_perturbed)
print('\nTakeaway: For conjugate Gaussian models, CAVI = exact Bayesian inference. '
      'The ELBO is a concave function of m, maximised at the posterior mean.')

---

## Exercise 5 ★★  Metropolis-Hastings on a Bimodal Distribution

The Metropolis-Hastings (MH) algorithm samples from an unnormalised target
$\tilde{p}(x) = \exp(-x^2/2) + \exp(-(x-4)^2/2)$  (a bimodal mixture).

**Parts:**

(a) Implement one step of MH with Gaussian proposal $q(x'|x) = \mathcal{N}(x, \sigma^2)$.
    Acceptance ratio: $\alpha = \min\!\left(1,\,\frac{\tilde{p}(x')}{\tilde{p}(x)}\right)$ (symmetric proposal).

(b) Run 10 000 iterations from $x_0 = 0$ with $\sigma = 0.5$ and $\sigma = 3.0$.
    Compute the empirical acceptance rate for each.

(c) Estimate $\mathbb{E}[x]$ and $\mathbb{E}[x^2]$ from each chain (discard first 1000 as burn-in).
    True values (from normalisation): $\mathbb{E}[x] \approx 2.0$, $\mathbb{E}[x^2] \approx 8.0$.

(d) Explain why $\sigma = 0.5$ might get stuck and $\sigma = 3.0$ mixes better for this target.


In [ ]:
# Exercise 5: Your Solution
import numpy as np

np.random.seed(42)

def log_target(x):
    # YOUR CODE HERE
    pass

def mh_chain(n_steps, sigma, x0=0.0):
    # YOUR CODE HERE — return (samples array, acceptance_rate)
    pass

samples_small, acc_small = mh_chain(10000, sigma=0.5)
samples_large, acc_large = mh_chain(10000, sigma=3.0)
print('acceptance small sigma:', acc_small)
print('acceptance large sigma:', acc_large)


In [ ]:
# Exercise 5: Solution
import numpy as np
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-1):
    ok = abs(got - expected) < tol
    print(f"{'PASS' if ok else 'FAIL'} — {name}: got {got:.4f}, expected {expected:.4f}")
    return ok

def log_target(x):
    return np.logaddexp(-x**2/2, -(x-4)**2/2)

def mh_chain(n_steps, sigma, x0=0.0):
    x = x0
    samples = np.zeros(n_steps)
    accepted = 0
    for t in range(n_steps):
        x_prop = x + sigma * np.random.randn()
        log_alpha = log_target(x_prop) - log_target(x)
        if np.log(np.random.rand()) < log_alpha:
            x = x_prop
            accepted += 1
        samples[t] = x
    return samples, accepted / n_steps

samples_small, acc_small = mh_chain(10000, sigma=0.5)
samples_large, acc_large = mh_chain(10000, sigma=3.0)

burn = 1000
ex_small  = samples_small[burn:].mean()
ex2_small = (samples_small[burn:]**2).mean()
ex_large  = samples_large[burn:].mean()
ex2_large = (samples_large[burn:]**2).mean()

header('Exercise 5: Metropolis-Hastings')
print(f'sigma=0.5: acc={acc_small:.3f}, E[x]={ex_small:.3f}, E[x^2]={ex2_small:.3f}')
print(f'sigma=3.0: acc={acc_large:.3f}, E[x]={ex_large:.3f}, E[x^2]={ex2_large:.3f}')
check_close('E[x] large sigma ~ 2.0', ex_large, 2.0, tol=0.5)
check_close('E[x^2] large sigma ~ 8.0', ex2_large, 8.0, tol=1.5)
print('\nTakeaway: Proposal std must match the scale of the target; too small => slow mixing, too large => high rejection.')


---

## Exercise 6 ★★  Gaussian Process Regression

Implement GP regression with an RBF kernel
$k(x,x') = \sigma_f^2 \exp\!\left(-\frac{(x-x')^2}{2\ell^2}\right)$.

**Parts:**

(a) Implement the RBF kernel matrix $K_{ij} = k(x_i, x_j)$.

(b) Given $n=8$ noisy observations $(X_{\text{tr}}, \mathbf{y})$ with noise $\sigma_n^2$,
    compute the posterior mean and variance at test points $X_*$:

$$\boldsymbol{\mu}_* = K_*^\top (K + \sigma_n^2 I)^{-1} \mathbf{y}$$
$$\Sigma_* = K_{**} - K_*^\top (K + \sigma_n^2 I)^{-1} K_*$$

(c) Verify that training point predictive means match observations within $2\sigma_n$.

(d) Compute the log marginal likelihood
    $\log p(\mathbf{y}) = -\frac{1}{2}\mathbf{y}^\top C^{-1}\mathbf{y} - \frac{1}{2}\log|C| - \frac{n}{2}\log 2\pi$
    where $C = K + \sigma_n^2 I$.


In [ ]:
# Exercise 6: Your Solution
import numpy as np
np.random.seed(42)

def rbf_kernel(X1, X2, sigma_f=1.0, ell=1.0):
    # YOUR CODE HERE — return shape (len(X1), len(X2))
    pass

def gp_posterior(X_tr, y_tr, X_test, sigma_f=1.0, ell=1.0, sigma_n=0.1):
    # YOUR CODE HERE — return (mu_star, var_star)
    pass

def log_marginal_likelihood(X_tr, y_tr, sigma_f=1.0, ell=1.0, sigma_n=0.1):
    # YOUR CODE HERE
    pass

np.random.seed(0)
X_tr = np.linspace(0, 5, 8)
y_tr = np.sin(X_tr) + 0.1 * np.random.randn(8)
X_test = np.linspace(0, 5, 50)
mu, var = gp_posterior(X_tr, y_tr, X_test)
print('Posterior mean at first 3 test points:', mu[:3])
print('Posterior var at first 3 test points:', var[:3])


In [ ]:
# Exercise 6: Solution
import numpy as np
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def rbf_kernel(X1, X2, sigma_f=1.0, ell=1.0):
    X1 = np.array(X1).reshape(-1, 1)
    X2 = np.array(X2).reshape(-1, 1)
    sq_dist = ((X1 - X2.T)**2)
    return sigma_f**2 * np.exp(-sq_dist / (2 * ell**2))

def gp_posterior(X_tr, y_tr, X_test, sigma_f=1.0, ell=1.0, sigma_n=0.1):
    n = len(X_tr)
    K    = rbf_kernel(X_tr, X_tr, sigma_f, ell)
    K_s  = rbf_kernel(X_tr, X_test, sigma_f, ell)
    K_ss = rbf_kernel(X_test, X_test, sigma_f, ell)
    C = K + sigma_n**2 * np.eye(n)
    L = np.linalg.cholesky(C)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_tr))
    mu_star = K_s.T @ alpha
    v = np.linalg.solve(L, K_s)
    var_star = np.diag(K_ss) - np.sum(v**2, axis=0)
    return mu_star, var_star

def log_marginal_likelihood(X_tr, y_tr, sigma_f=1.0, ell=1.0, sigma_n=0.1):
    n = len(X_tr)
    K = rbf_kernel(X_tr, X_tr, sigma_f, ell)
    C = K + sigma_n**2 * np.eye(n)
    L = np.linalg.cholesky(C)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_tr))
    lml = -0.5 * y_tr @ alpha - np.sum(np.log(np.diag(L))) - 0.5 * n * np.log(2*np.pi)
    return lml

np.random.seed(0)
X_tr = np.linspace(0, 5, 8)
y_tr = np.sin(X_tr) + 0.1 * np.random.randn(8)
X_test = np.linspace(0, 5, 50)

mu, var = gp_posterior(X_tr, y_tr, X_test)
mu_tr, var_tr = gp_posterior(X_tr, y_tr, X_tr)
lml = log_marginal_likelihood(X_tr, y_tr)

header('Exercise 6: GP Regression')
print(f'Log marginal likelihood: {lml:.4f}')
check_true('posterior variance non-negative', np.all(var >= -1e-10))
residuals = np.abs(mu_tr - y_tr)
sigma_n = 0.1
check_true('training mean within 3sigma_n', np.all(residuals < 3 * sigma_n + 3 * np.sqrt(var_tr + 1e-12)))
print('\nTakeaway: GP posterior is exact and closed-form; log marginal likelihood enables kernel hyperparameter tuning.')


---

## Exercise 7 ★★  VAE ELBO and Reparameterisation

The VAE objective is the ELBO:

$$\mathcal{L} = \mathbb{E}_{q_\phi(\mathbf{z}|\mathbf{x})}[\log p_\theta(\mathbf{x}|\mathbf{z})]
- D_{\mathrm{KL}}(q_\phi(\mathbf{z}|\mathbf{x}) \,\|\, p(\mathbf{z}))$$

With $q_\phi = \mathcal{N}(\boldsymbol{\mu}, \operatorname{diag}(\boldsymbol{\sigma}^2))$ and prior $p(\mathbf{z}) = \mathcal{N}(\mathbf{0}, I)$:

$$D_{\mathrm{KL}} = \frac{1}{2}\sum_{j=1}^d \left(\mu_j^2 + \sigma_j^2 - \log\sigma_j^2 - 1\right)$$

**Parts:**

(a) Implement `kl_gaussian_standard(mu, log_var)` using the closed-form above
    (encoder outputs $\log\sigma^2$ for numerical stability).

(b) Implement the reparameterisation trick: sample
    $\mathbf{z} = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\varepsilon}$,
    $\boldsymbol{\varepsilon} \sim \mathcal{N}(\mathbf{0},I)$.

(c) Verify: if $\mu = \mathbf{0}$, $\log\sigma^2 = \mathbf{0}$ ($\sigma = 1$), then $D_{\mathrm{KL}} = 0$.

(d) Verify: if $\mu = [3,0]$, $\log\sigma^2 = [-2, 0]$, the reparameterised
    samples have the correct empirical mean and variance.


In [ ]:
# Exercise 7: Your Solution
import numpy as np
np.random.seed(42)

def kl_gaussian_standard(mu, log_var):
    # YOUR CODE HERE — sum over latent dims, return scalar
    pass

def reparameterise(mu, log_var, n_samples=1000):
    # YOUR CODE HERE — return shape (n_samples, d)
    pass

mu1 = np.zeros(2)
lv1 = np.zeros(2)
print('KL (should be 0):', kl_gaussian_standard(mu1, lv1))

mu2 = np.array([3.0, 0.0])
lv2 = np.array([-2.0, 0.0])
z = reparameterise(mu2, lv2)
print('Empirical mean:', z.mean(axis=0))
print('Empirical std: ', z.std(axis=0))


In [ ]:
# Exercise 7: Solution
import numpy as np
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-6):
    ok = np.allclose(got, expected, atol=tol)
    print(f"{'PASS' if cond else 'FAIL'} — {name}".replace("cond", str(ok)))

def kl_gaussian_standard(mu, log_var):
    return 0.5 * np.sum(mu**2 + np.exp(log_var) - log_var - 1)

def reparameterise(mu, log_var, n_samples=1000):
    eps = np.random.randn(n_samples, len(mu))
    return mu + np.exp(0.5 * log_var) * eps

mu1, lv1 = np.zeros(2), np.zeros(2)
mu2 = np.array([3.0, 0.0])
lv2 = np.array([-2.0, 0.0])

kl0 = kl_gaussian_standard(mu1, lv1)
kl2 = kl_gaussian_standard(mu2, lv2)
z   = reparameterise(mu2, lv2, n_samples=50000)

header('Exercise 7: VAE ELBO and Reparameterisation')
print(f'KL(N(0,I) || N(0,I)) = {kl0:.6f}  (expected 0)')
ok_kl0 = abs(kl0) < 1e-9
print(f"{'PASS' if ok_kl0 else 'FAIL'} — KL is zero for unit normal")
expected_std = np.exp(0.5 * lv2)
ok_mean = np.allclose(z.mean(0), mu2, atol=0.05)
ok_std  = np.allclose(z.std(0),  expected_std, atol=0.05)
print(f"{'PASS' if ok_mean else 'FAIL'} — empirical mean ~ [{mu2[0]:.1f}, {mu2[1]:.1f}]")
print(f"{'PASS' if ok_std  else 'FAIL'} — empirical std  ~ [{expected_std[0]:.3f}, {expected_std[1]:.3f}]")
print(f'KL(q||p) = {kl2:.4f}')
print('\nTakeaway: Reparameterisation makes z differentiable w.r.t. phi; KL closed form avoids Monte Carlo for the regulariser.')


---

## Exercise 8 ★★  HMM Forward Algorithm

A Hidden Markov Model has $K = 2$ hidden states, $V = 3$ observations, and $T = 5$ time steps.

**Parts:**

(a) Implement the forward algorithm in log-space to compute $\log p(\mathbf{o}_{1:T})$.

    Forward variable: $\alpha_t(k) = p(o_{1:t}, s_t = k)$

    Recursion: $\alpha_t(k) = \left[\sum_j \alpha_{t-1}(j)\, A_{jk}\right] B_{k,o_t}$

(b) Verify your log-likelihood against brute-force summation over all $K^T = 32$ state sequences.

(c) Implement the Viterbi algorithm to find the most likely state sequence.

(d) Check that the Viterbi path has higher joint probability than 5 random state sequences.


In [ ]:
# Exercise 8: Your Solution
import numpy as np
np.random.seed(42)

# HMM parameters
pi = np.array([0.6, 0.4])          # initial state probabilities
A  = np.array([[0.7, 0.3],         # transition matrix
               [0.4, 0.6]])
B  = np.array([[0.5, 0.4, 0.1],    # emission matrix (K x V)
               [0.1, 0.3, 0.6]])
obs = np.array([0, 1, 2, 1, 0])    # observed sequence (0-indexed)

def forward_log(obs, pi, A, B):
    # YOUR CODE HERE — return log p(obs)
    pass

def viterbi(obs, pi, A, B):
    # YOUR CODE HERE — return best_path (list of state indices)
    pass

log_p = forward_log(obs, pi, A, B)
path  = viterbi(obs, pi, A, B)
print('log p(obs):', log_p)
print('Viterbi path:', path)


In [ ]:
# Exercise 8: Solution
import numpy as np
from itertools import product
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def logsumexp(a):
    m = a.max()
    return m + np.log(np.sum(np.exp(a - m)))

def check_close(name, got, expected, tol=1e-6):
    ok = abs(got - expected) < tol
    print(f"{'PASS' if ok else 'FAIL'} — {name}: got {got:.6f}, expected {expected:.6f}")
    return ok

pi = np.array([0.6, 0.4])
A  = np.array([[0.7, 0.3], [0.4, 0.6]])
B  = np.array([[0.5, 0.4, 0.1], [0.1, 0.3, 0.6]])
obs = np.array([0, 1, 2, 1, 0])
K, T = 2, len(obs)

def forward_log(obs, pi, A, B):
    T = len(obs)
    K = len(pi)
    log_alpha = np.log(pi) + np.log(B[:, obs[0]])
    for t in range(1, T):
        log_alpha_new = np.zeros(K)
        for k in range(K):
            log_alpha_new[k] = logsumexp(log_alpha + np.log(A[:, k])) + np.log(B[k, obs[t]])
        log_alpha = log_alpha_new
    return logsumexp(log_alpha)

def viterbi(obs, pi, A, B):
    T = len(obs)
    K = len(pi)
    log_delta = np.log(pi) + np.log(B[:, obs[0]])
    psi = np.zeros((T, K), dtype=int)
    for t in range(1, T):
        scores = log_delta[:, None] + np.log(A)
        psi[t] = scores.argmax(0)
        log_delta = scores.max(0) + np.log(B[:, obs[t]])
    path = [int(log_delta.argmax())]
    for t in range(T-1, 0, -1):
        path.append(psi[t, path[-1]])
    return list(reversed(path))

def brute_force(obs, pi, A, B):
    total = 0.0
    for seq in product(range(K), repeat=T):
        p = pi[seq[0]] * B[seq[0], obs[0]]
        for t in range(1, T):
            p *= A[seq[t-1], seq[t]] * B[seq[t], obs[t]]
        total += p
    return np.log(total)

log_p = forward_log(obs, pi, A, B)
log_p_bf = brute_force(obs, pi, A, B)
path = viterbi(obs, pi, A, B)

header('Exercise 8: HMM Forward + Viterbi')
check_close('forward == brute force', log_p, log_p_bf, tol=1e-10)

def joint_log_prob(seq, obs, pi, A, B):
    p = np.log(pi[seq[0]]) + np.log(B[seq[0], obs[0]])
    for t in range(1, len(obs)):
        p += np.log(A[seq[t-1], seq[t]]) + np.log(B[seq[t], obs[t]])
    return p

viterbi_lp = joint_log_prob(path, obs, pi, A, B)
random_lps = [joint_log_prob(list(np.random.randint(0,K,T)), obs, pi, A, B) for _ in range(10)]
ok = all(viterbi_lp >= lp for lp in random_lps)
print(f"{'PASS' if ok else 'FAIL'} — Viterbi path has highest joint log-prob")
print(f'Viterbi path: {path}')
print('\nTakeaway: Forward algorithm runs in O(T*K^2) instead of O(K^T); log-space prevents underflow for long sequences.')


---

## Exercise 9 ★★★  Normalising Flow: Change of Variables

A normalising flow transforms a simple base density $p_Z(\mathbf{z})$ through
an invertible map $f: \mathbf{z} \mapsto \mathbf{x}$:

$$\log p_X(\mathbf{x}) = \log p_Z(f^{-1}(\mathbf{x})) + \log\left|\det J_{f^{-1}}(\mathbf{x})\right|$$

Implement a simple affine coupling layer:

$$\mathbf{x}_{1:d/2} = \mathbf{z}_{1:d/2}, \quad
\mathbf{x}_{d/2+1:d} = \mathbf{z}_{d/2+1:d} \odot \exp(s(\mathbf{z}_{1:d/2})) + t(\mathbf{z}_{1:d/2})$$

**Parts:**

(a) Implement `forward(z)` → $\mathbf{x}$ and `inverse(x)` → $\mathbf{z}$.

(b) Implement `log_det_jacobian(z)` — for a coupling layer it equals $\sum_j s_j(\mathbf{z}_{1:d/2})$.

(c) Verify the cycle-consistency: `inverse(forward(z)) ≈ z` for random inputs.

(d) Estimate $\mathbb{E}[\log p_X(\mathbf{x})]$ under the flow using 1000 samples from
    $\mathbf{z} \sim \mathcal{N}(\mathbf{0}, I)$ and linear $s, t$ networks.


In [ ]:
# Exercise 9: Your Solution
import numpy as np
np.random.seed(42)

d = 4  # latent dimension (must be even)

# Simple linear scale/shift networks for the coupling layer
Ws = np.random.randn(d//2, d//2) * 0.1
Wt = np.random.randn(d//2, d//2) * 0.1

def scale_net(z1):  return z1 @ Ws   # s(z_1:d/2)
def shift_net(z1):  return z1 @ Wt   # t(z_1:d/2)

def forward(z):
    # YOUR CODE HERE — return x
    pass

def inverse(x):
    # YOUR CODE HERE — return z
    pass

def log_det_jacobian(z):
    # YOUR CODE HERE — return scalar
    pass

z_test = np.random.randn(5, d)
x_test = forward(z_test)
z_rec  = inverse(x_test)
print('Cycle error:', np.abs(z_rec - z_test).max())


In [ ]:
# Exercise 9: Solution
import numpy as np
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

d = 4
np.random.seed(0)
Ws = np.random.randn(d//2, d//2) * 0.1
Wt = np.random.randn(d//2, d//2) * 0.1

def scale_net(z1): return z1 @ Ws
def shift_net(z1): return z1 @ Wt

def forward(z):
    z1, z2 = z[:, :d//2], z[:, d//2:]
    s = scale_net(z1)
    t = shift_net(z1)
    x2 = z2 * np.exp(s) + t
    return np.concatenate([z1, x2], axis=1)

def inverse(x):
    x1, x2 = x[:, :d//2], x[:, d//2:]
    s = scale_net(x1)
    t = shift_net(x1)
    z2 = (x2 - t) * np.exp(-s)
    return np.concatenate([x1, z2], axis=1)

def log_det_jacobian(z):
    z1 = z[:, :d//2]
    s = scale_net(z1)
    return s.sum(axis=1)

def log_normal(x): return -0.5 * (x**2).sum(1) - 0.5 * d * np.log(2*np.pi)

np.random.seed(42)
z_test = np.random.randn(5, d)
x_test = forward(z_test)
z_rec  = inverse(x_test)
cycle_err = np.abs(z_rec - z_test).max()

z_samp = np.random.randn(1000, d)
x_samp = forward(z_samp)
log_px = log_normal(z_samp) - log_det_jacobian(z_samp)

header('Exercise 9: Normalising Flow Coupling Layer')
check_true('cycle consistency < 1e-12', cycle_err < 1e-12)
check_true('log det jacobian is finite', np.isfinite(log_det_jacobian(z_samp)).all())
print(f'Mean log p_X(x): {log_px.mean():.4f}')
print('\nTakeaway: Coupling layers allow exact density evaluation in O(d); triangular Jacobian makes det cheap to compute.')


---

## Exercise 10 ★★★  In-Context Learning as Bayesian Inference

Xie et al. (2021) show that ICL approximates Bayesian posterior predictive
inference under a latent concept $\theta$:

$$p(y_{n+1} | x_{n+1}, \mathcal{D}_n) = \int p(y_{n+1}|x_{n+1},\theta)\,p(\theta|\mathcal{D}_n)\,d\theta$$

Simulate this with a **Beta-Bernoulli** ICL scenario:

- Latent concept: coin bias $\theta \sim \text{Beta}(\alpha_0, \beta_0)$
- Likelihood: $y_i | \theta \sim \text{Bernoulli}(\theta)$
- Posterior after $n$ observations: $\theta | \mathcal{D}_n \sim \text{Beta}(\alpha_0 + H, \beta_0 + T)$

**Parts:**

(a) Implement `bayes_icl_predict(alpha0, beta0, heads, tails)` that returns
    the posterior predictive $p(y=1 | \mathcal{D}_n) = \frac{\alpha_0 + H}{\alpha_0 + H + \beta_0 + T}$.

(b) Simulate a "transformer oracle" that sees $n \in \{0, 1, 5, 20, 100\}$ examples
    from a coin with true $\theta^* = 0.7$, computes the Bayesian prediction, and
    compare against the MLE prediction $\hat{\theta}_{\text{MLE}} = H/(H+T)$ (undefined at $n=0$).

(c) Compute and plot the KL divergence
    $D_{\mathrm{KL}}(\text{Beta}(\alpha_n, \beta_n) \| \text{Beta}(\alpha_0, \beta_0))$
    as $n$ increases — it should grow monotonically (posterior moves away from prior).

(d) What happens when the prior is misspecified ($\alpha_0 = \beta_0 = 10$, strong
    prior toward 0.5, true $\theta^* = 0.9$)? How many examples are needed to
    correct the posterior prediction to within 0.05 of the truth?


In [ ]:
# Exercise 10: Your Solution
import numpy as np
from scipy.special import betaln
np.random.seed(42)

def bayes_icl_predict(alpha0, beta0, heads, tails):
    # YOUR CODE HERE — return p(y=1 | D_n)
    pass

def kl_beta(a1, b1, a2, b2):
    # KL(Beta(a1,b1) || Beta(a2,b2)) closed form
    # YOUR CODE HERE
    pass

true_theta = 0.7
alpha0, beta0 = 1.0, 1.0
for n in [0, 1, 5, 20, 100]:
    H = int(round(n * true_theta))
    T = n - H
    pred = bayes_icl_predict(alpha0, beta0, H, T)
    print(f'n={n:3d}: Bayes={pred:.4f}')


In [ ]:
# Exercise 10: Solution
import numpy as np
from scipy.special import betaln, digamma
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def bayes_icl_predict(alpha0, beta0, heads, tails):
    alpha_n = alpha0 + heads
    beta_n  = beta0 + tails
    return alpha_n / (alpha_n + beta_n)

def kl_beta(a1, b1, a2, b2):
    # KL(Beta(a1,b1) || Beta(a2,b2))
    return (betaln(a2, b2) - betaln(a1, b1)
            + (a1 - a2) * digamma(a1)
            + (b1 - b2) * digamma(b1)
            + (a2 - a1 + b2 - b1) * digamma(a1 + b1))

true_theta = 0.7
alpha0, beta0 = 1.0, 1.0
ns = [0, 1, 5, 20, 50, 100]

header('Exercise 10: ICL as Bayesian Inference')
print(f'True theta = {true_theta}')
print(f'{'n':>5}  {'Bayes':>8}  {'MLE':>8}  {'|B-truth|':>10}  KL_from_prior')
kl_prev = 0.0
kl_vals = []
for n in ns:
    H = int(round(n * true_theta))
    T = n - H
    pred  = bayes_icl_predict(alpha0, beta0, H, T)
    mle   = H / n if n > 0 else float('nan')
    kl    = kl_beta(alpha0 + H, beta0 + T, alpha0, beta0)
    kl_vals.append(kl)
    print(f'{n:>5}  {pred:>8.4f}  {mle:>8.4f}  {abs(pred - true_theta):>10.4f}  {kl:.4f}')

check_true('KL monotonically increases', all(kl_vals[i] <= kl_vals[i+1] for i in range(len(kl_vals)-1)))

# Misspecification: strong prior toward 0.5, true theta = 0.9
alpha_mis, beta_mis = 10.0, 10.0
true_mis = 0.9
n_correct = None
for n in range(1, 500):
    H = int(round(n * true_mis))
    T = n - H
    pred = bayes_icl_predict(alpha_mis, beta_mis, H, T)
    if abs(pred - true_mis) < 0.05:
        n_correct = n
        break
print(f'\nMisspecification (prior alpha=beta=10, true theta=0.9):')
print(f'Examples to correct to within 0.05: {n_correct}')
check_true('converges in < 200 examples', n_correct is not None and n_correct < 200)
print('\nTakeaway: ICL is Bayesian posterior predictive inference; strong misspecified priors require O(prior_strength) examples to overcome.')


---

## What to Review After Finishing

- [ ] **Conjugacy** — can you derive the posterior update for Beta-Bernoulli and Gaussian-Gaussian from scratch?
- [ ] **Exponential family** — can you identify $\eta$, $T$, $A$, and $h$ for Gaussian, Poisson, Bernoulli, and Gamma?
- [ ] **EM** — can you show that the ELBO is a lower bound on $\log p(\mathbf{x})$ using Jensen's inequality?
- [ ] **CAVI** — can you derive the coordinate update $q_j^*(z_j) \propto \exp(\mathbb{E}_{-j}[\log p])$?
- [ ] **Reparameterisation** — why does score-function estimation have high variance? Why does reparam fix it?
- [ ] **MH acceptance** — what does detailed balance mean, and why does the MH acceptance ratio enforce it?
- [ ] **GP regression** — why is the posterior mean a weighted sum of kernel evaluations at training points?
- [ ] **Forward algorithm** — why does log-space computation prevent numerical underflow for long sequences?
- [ ] **Coupling layers** — why is the Jacobian of a coupling layer triangular, and what does that buy you?
- [ ] **ICL-Bayes link** — in what sense does a transformer trained on the prior approximate Bayesian inference?

## References

1. Bishop (2006) *Pattern Recognition and Machine Learning* — Chapters 2–4, 9, 12
2. Murphy (2022) *Probabilistic Machine Learning* — Chapters 3–6, 18, 20, 23
3. Gelman et al. (2013) *Bayesian Data Analysis* — Chapters 1–5, 11–12
4. Kingma & Welling (2013) *Auto-Encoding Variational Bayes* — arXiv:1312.6114
5. Xie et al. (2021) *An Explanation of In-Context Learning as Implicit Bayesian Inference* — arXiv:2111.02080
6. Song et al. (2021) *Score-Based Generative Modeling through SDEs* — arXiv:2011.13456
7. Lipman et al. (2022) *Flow Matching for Generative Modeling* — arXiv:2210.02747
8. Rasmussen & Williams (2006) *Gaussian Processes for Machine Learning* — MIT Press (free online)
